In [ ]:
# Lab type: debug
# Course: AI403 — Building Production AI Agents
# Lesson: Persistent Memory: What to Store, Where It Came From, When to Re-Check It
# Task: The invoice-routing agent's memory sends invoices to the wrong people. There are 3 bugs.
# After fixing each bug, write a one-sentence explanation in the comment cell below it.

# Lab: The Approver Who Left in July

Nimbus Analytics resells to several tenants. An agent routes each customer's invoices to that customer's approver, and remembers approvers between runs so it doesn't have to look them up every time. It worked in March. It's now September.

**Outputs are cleared.** Run every cell top to bottom.

## Setup

In [ ]:
# No installs needed: this lab uses the Python standard library only.

In [ ]:
from datetime import date

# Systems of record. Two different tenants each have a customer called "Acme Corp".
APPROVERS = {("t1", "Acme Corp"): "priya.shah@acme.example",
             ("t2", "Acme Corp"): "j.ruiz@acme-industrial.example",
             ("t1", "Initech"): "finance@initech.example"}
LOOKUPS = {"count": 0}

def lookup_approver(tenant, customer):
    """The approvals system: always current, but slow and rate-limited."""
    LOOKUPS["count"] += 1
    return APPROVERS[(tenant, customer)]

def priya_leaves():
    """1 August 2026: Priya leaves Acme; the approvals system is updated."""
    APPROVERS[("t1", "Acme Corp")] = "tom.okafor@acme.example"

print(lookup_approver("t1", "Acme Corp"))

## The memory code under review

In [ ]:
# --- GENERATED MEMORY + ROUTING (review this code — is it correct?) ---
memory = {}

def remember(key, value):
    memory[key] = value                                   # <- what is stored?

def route_invoice(tenant, customer, today):
    key = f"approver:{customer}"                          # <- whose memory is this?
    if key in memory:
        return memory[key]                                # <- used for an action as-is
    approver = lookup_approver(tenant, customer)
    remember(key, approver)
    return approver

# March: first invoices
print("2026-03-04 t1 Acme:", route_invoice("t1", "Acme Corp", date(2026, 3, 4)))
print("2026-03-04 t1 Initech:", route_invoice("t1", "Initech", date(2026, 3, 4)))
priya_leaves()
# September
print("2026-09-30 t1 Acme:", route_invoice("t1", "Acme Corp", date(2026, 9, 30)))
print("2026-09-30 t2 Acme:", route_invoice("t2", "Acme Corp", date(2026, 9, 30)))
print("lookups made:", LOOKUPS["count"])

**Question 1.** Two September invoices went to the wrong person. Who received each, and who should have? Name the three bugs in the code above (look at the three `<-` markers).

In [ ]:
# Your answer (as comments):

<details>
<summary>🔑 Reveal answer — Question 1</summary>

- t1's Acme invoice went to **Priya**, who left in July; it should have gone to **Tom**. Memory handed back a March fact with no sign of its age.
- t2's Acme invoice went to **Priya** too — an employee of a *different company in a different tenant* — because the memory key is the customer name only. It should have gone to **j.ruiz@acme-industrial.example**.

Bugs: (1) `remember` stores a bare value — no source, no date, so nothing can expire or audit it; (2) `route_invoice` uses memory directly for an action instead of re-checking the system of record; (3) the key isn't scoped by tenant, so one tenant's memory is recalled for another (OWASP MCP10:2025, context over-sharing).

</details>

## Fix the store

**Question 2.** Write a `MemoryRecord` dataclass with `key`, `value`, `source`, `observed` (date), `ttl_days` and `affects_actions`, plus an `is_stale(today)` method. Rewrite `remember(tenant, customer, value, source, today)` to store a record under a key that includes the tenant.

In [ ]:
# Work here: MemoryRecord and a tenant-scoped remember

In [ ]:
# Bug 1 explanation:
# Bug 3 explanation:

<details>
<summary>🔑 Reveal answer — Question 2</summary>

```python
from dataclasses import dataclass

@dataclass
class MemoryRecord:
    key: tuple
    value: str
    source: str
    observed: date
    ttl_days: int
    affects_actions: bool

    def is_stale(self, today):
        return (today - self.observed).days > self.ttl_days

memory = {}

def remember(tenant, customer, value, source, today, ttl_days=30):
    key = ("approver", tenant, customer)                  # scoped like data
    memory[key] = MemoryRecord(key, value, source, today, ttl_days, affects_actions=True)
```

*Bug 1:* a bare value can't be expired, audited or corrected because nothing records where or when it was learned. *Bug 3:* memory must be scoped exactly like the data it came from, or it leaks across tenants.

</details>

**Question 3.** Rewrite `route_invoice(tenant, customer, today)`: memory may say *where to look*, but because routing is an action, the approver is re-checked with `lookup_approver` before use, and the memory is corrected if it differs. Reset the world with the cell provided, replay March and September, and confirm both September invoices go to the right people.

In [ ]:
# Run this to reset the world before replaying
APPROVERS[("t1", "Acme Corp")] = "priya.shah@acme.example"
LOOKUPS["count"] = 0
memory = {}

In [ ]:
# Work here: route_invoice with re-check, then replay

In [ ]:
# Bug 2 explanation:

<details>
<summary>🔑 Reveal answer — Question 3</summary>

```python
def route_invoice(tenant, customer, today):
    key = ("approver", tenant, customer)
    record = memory.get(key)
    if record and not record.affects_actions and not record.is_stale(today):
        return record.value, "from memory"
    current = lookup_approver(tenant, customer)          # the system of record decides
    note = "looked up"
    if record and record.value != current:
        note = f"memory was wrong (said {record.value}); corrected"
    remember(tenant, customer, current, "approvals system", today)
    return current, note

APPROVERS[("t1", "Acme Corp")] = "priya.shah@acme.example"
LOOKUPS["count"], memory = 0, {}
print(route_invoice("t1", "Acme Corp", date(2026, 3, 4)))
priya_leaves()
print(route_invoice("t1", "Acme Corp", date(2026, 9, 30)))
print(route_invoice("t2", "Acme Corp", date(2026, 9, 30)))
```

*Bug 2:* for a fact that drives an action, memory is a cache and the system of record wins. You pay a lookup per invoice — cheaper than one invoice sent to someone who left. (For facts that only shape wording, the TTL path lets memory be used directly.)

</details>

**Question 4.** Write a stale-fact eval: plant a memory record for `("t1", "Initech")` with a wrong value dated a year ago, change nothing else, and assert `route_invoice` returns the system-of-record value. Add a second case asserting that t2 never receives a t1 memory.

In [ ]:
# Work here: two asserts

<details>
<summary>🔑 Reveal answer — Question 4</summary>

```python
memory[("approver", "t1", "Initech")] = MemoryRecord(
    ("approver", "t1", "Initech"), "old.cfo@initech.example", "email 2025-09-01",
    date(2025, 9, 1), 30, True)
value, _ = route_invoice("t1", "Initech", date(2026, 9, 30))
assert value == APPROVERS[("t1", "Initech")], value

value, _ = route_invoice("t2", "Acme Corp", date(2026, 9, 30))
assert value == APPROVERS[("t2", "Acme Corp")], value
print("stale-fact and tenant-scope evals pass")
```

"The agent seems fine" never catches this, because stale memory produces fluent, confident output. The eval has to plant the stale fact deliberately.

</details>

**Question 5 (judgement).** A teammate proposes also remembering each customer's billing API token so the agent can skip the auth step. What do you say?

In [ ]:
# Your answer (as a comment):

<details>
<summary>🔑 Reveal answer — Question 5</summary>

No. Secrets written into memory end up in prompts, logs and traces, and anything that can read memory — including an injected instruction — can read them (OWASP MCP01:2025 lists secrets stored in model memory). Credentials belong in the tool's own configuration, scoped per tenant and short-lived; the model should never see them.

</details>

## Summary

1. Every memory record needs a _______ and a last-verified date.
2. For facts that drive actions, memory is a cache and the _______ wins.
3. Memory must be scoped like the data it came from, or it leaks across _______.

<details>
<summary>🔑 Reveal summary answers</summary>

1. **source**
2. **system of record**
3. **tenants (users)**

</details>